# Activation Steering with Linear Probe

Re-runs the problems the model **failed** on (from `traces_all.h5`) with probe-guided activation steering.

## Method

1. Compute steering vector `v = mean(h+) - mean(h-)` from all collected hidden states in `traces_all.h5`.
2. Load the fine-tuned linear probe (`linear_probe.pt`).
3. For each failed problem, generate a new response with two sets of hooks:
   - **Probe hook** on `layers[-1]` (layer 27): at every `\n\n`, reads the hidden state and runs the probe. If `P(correct) < STEER_THRESHOLD`, sets a steer flag for the next step.
   - **Steer hooks** on `STEER_LAYERS` (middle-to-late layers, default 14–21): on the *next* forward step after the flag is set, adds `v × ALPHA` to each target layer's hidden state. The perturbation propagates forward through layers 22–27, reshaping the model's higher-level reasoning before reaching the LM head.
4. Report per-problem token counts and overall accuracy vs. the original.

## Why separate read and write layers

Steering at the last layer (`layers[-1]`) only biases the immediate next-token logits — a one-step nudge with no propagation. Steering at earlier layers lets the perturbation ripple forward through several transformer blocks, influencing the model's internal representations over multiple tokens. That's where the real effect on reasoning strategy comes from.

The probe still **reads** from layer 27 (where it was trained), but **writes** into layers 14–21 so the correction has room to propagate.

## Steering vector intuition

If `h+ ≈ h- + c`, then `v ≈ c` — the exact translation from failing to succeeding activations.
Adding `v × α` to a low-confidence hidden state moves it geometrically toward where the model's
activations live when it solves problems correctly.

**Alpha calibration** (measured against the layer-27 probe as a proxy):
- `α = 0.3` → logit shift ≈ 0.8 → bumps P(correct) from 0.35 → ~0.56
- `α = 1.0` → logit shift ≈ 2.7 → bumps P(correct) from 0.35 → ~0.89
- Default `ALPHA = 0.5` is a mild-to-moderate push.

Adjust `ALPHA`, `STEER_THRESHOLD`, and `STEER_LAYERS` in the Config cell.

In [ ]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/Avni2000/cs639-final-project.git"
    REPO_DIR = "/content/cs639-final-project"
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    %cd {REPO_DIR}/probe-baseline

    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "./split_workflow/outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
%pip install -q "transformers==4.45.0" "torch>=2.3" "datasets" "pandas" \
               "math-verify[antlr4_13_2]" "tqdm" "h5py"

## Config

In [ ]:
# --- Inputs ---
HDF5_PATH   = os.path.join(OUTPUT_DIR, "traces_all.h5")    # from shard_merge.ipynb
PROBE_PATH  = os.path.join(OUTPUT_DIR, "linear_probe.pt")  # from probe_train.ipynb
CSV_PATH    = "split_workflow/aime_historical.csv"          # AIME problem set

# --- Outputs ---
STEERING_RESULTS_PATH = os.path.join(OUTPUT_DIR, "steering_results.json")

# --- Model ---
MODEL_ID        = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
HIDDEN_DIM      = 3584
PROBE_LAYER     = -1              # layer to READ from (last layer, index 27)
MAX_NEW_TOKENS  = 8192

# --- Steering ---
# Layers to WRITE into. Must be earlier than PROBE_LAYER so the perturbation
# propagates forward through the remaining layers before the probe reads it.
# DeepSeek-R1-Distill-Qwen-7B has 28 layers (0-27); middle-to-late = 14-21.
STEER_LAYERS    = list(range(14, 22))   # layers 14, 15, ..., 21

STEER_THRESHOLD = 0.35            # steer when probe confidence < this
ALPHA           = 0.5             # scale factor for v; see calibration notes in intro
MAX_STEERS_PER_PROBLEM = 10       # cap to avoid runaway steering

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU required. Runtime -> Change runtime type -> GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Load probe and compute steering vector

In [ ]:
import torch
import torch.nn as nn
import h5py

assert os.path.exists(HDF5_PATH),  f"Missing {HDF5_PATH} — run shard_merge.ipynb first"
assert os.path.exists(PROBE_PATH), f"Missing {PROBE_PATH} — run probe_train.ipynb first"

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Linear probe ---
class LinearProbe(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(-1)

ckpt = torch.load(PROBE_PATH, map_location=device)
probe = LinearProbe(ckpt["hidden_dim"]).to(device)
probe.load_state_dict(ckpt["state_dict"])
probe.eval()

mu  = ckpt["norm_mean"].to(device)   # [HIDDEN_DIM]
sig = ckpt["norm_std"].to(device)    # [HIDDEN_DIM]

print(f"Probe loaded (best_val_auc={ckpt['best_val_auc']:.4f})")

# --- Steering vector: v = mean(h+) - mean(h-) over all stored hidden states ---
h_pos_parts, h_neg_parts = [], []
with h5py.File(HDF5_PATH, "r") as f:
    for key in sorted(f.keys()):
        if not key.startswith("problem_"):
            continue
        grp = f[key]
        hs = torch.from_numpy(grp["hidden_states"][...]).float()
        if hs.shape[0] == 0:
            continue
        label = int(grp["label"][()])
        if label == 1:
            h_pos_parts.append(hs)
        else:
            h_neg_parts.append(hs)

H_pos = torch.cat(h_pos_parts, dim=0)  # [N+, HIDDEN_DIM]
H_neg = torch.cat(h_neg_parts, dim=0)  # [N-, HIDDEN_DIM]
steering_vec = (H_pos.mean(0) - H_neg.mean(0)).to(device)  # [HIDDEN_DIM]

print(f"h+: {H_pos.shape}  h-: {H_neg.shape}")
print(f"steering_vec norm: {steering_vec.norm().item():.2f}")

# Sanity: what does alpha * v do to the probe logit?
# In normalized space: shift = w · (v * alpha / sig)
w = ckpt["state_dict"]["fc.weight"].squeeze(0).to(device)
logit_shift = (w * steering_vec.to(w.device) / sig).sum().item() * ALPHA
p_before = torch.sigmoid(torch.tensor(torch.log(torch.tensor(STEER_THRESHOLD / (1 - STEER_THRESHOLD)))))
import math
base_logit = math.log(STEER_THRESHOLD / (1 - STEER_THRESHOLD))
p_after = 1.0 / (1.0 + math.exp(-(base_logit + logit_shift)))
print(f"\nAlpha={ALPHA}: probe logit shifts by {logit_shift:.3f}")
print(f"  P(correct) at threshold: {STEER_THRESHOLD:.2f} → {p_after:.3f} after steering")

## Identify failed problems

In [ ]:
import pandas as pd

# Load AIME problem set
df = pd.read_csv(CSV_PATH)
problems = df[["Question", "Answer"]].dropna().reset_index(drop=True)

# Find failed problem IDs from the merged HDF5
failed_ids = []
with h5py.File(HDF5_PATH, "r") as f:
    for key in sorted(f.keys()):
        if not key.startswith("problem_"):
            continue
        label = int(f[key]["label"][()])
        if label == 0:
            failed_ids.append(int(key.split("_", 1)[1]))

failed_ids = sorted(failed_ids)
print(f"Problems originally failed: {len(failed_ids)}")
print(f"Example IDs: {failed_ids[:10]}")

## Load model

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")

## Helpers

In [ ]:
def build_prompt(problem_text: str) -> str:
    messages = [{
        "role": "user",
        "content": (
            "Solve the following AIME problem. "
            "Your final answer must be a non-negative integer (0-999), "
            "placed inside \\boxed{}.\n\n" + problem_text
        ),
    }]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def probe_confidence(h_raw: torch.Tensor) -> float:
    """Run probe on a single raw hidden state vector [HIDDEN_DIM]."""
    h = h_raw.to(device).float()
    h_norm = (h - mu) / sig
    logit = probe.fc(h_norm.unsqueeze(0)).squeeze()
    return torch.sigmoid(logit).item()


def generate_with_steering(prompt: str):
    """
    Run greedy generation with probe-gated activation steering.

    Two sets of hooks:
      Probe hook  (layers[-1])    — reads hidden state at every \\n\\n; sets
                                    state["should_steer"] if conf < threshold.
      Steer hooks (STEER_LAYERS)  — on the *next* forward step after the flag
                                    is set, add steering_vec * ALPHA to each
                                    target layer's output, then clear the flag
                                    after the last steer layer fires.

    The perturbation is written at layers 14-21 and propagates forward through
    layers 22-27 before reaching the LM head, reshaping reasoning rather than
    just nudging logits.

    Returns:
        generated_text (str), n_tokens (int), n_steers (int), steer_log (list)
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    prompt_len = input_ids.shape[1]

    state = {
        "generated_ids":  [],
        "is_prompt_pass": True,   # skips all hooks on the first (full-prompt) forward pass
        "prev_text":      "",
        "should_steer":   False,  # set by probe hook; consumed by last steer hook
        "steer_count":    0,
        "steer_log":      [],     # {token_pos, conf_before}
    }

    # --- embed hook: track generated tokens, detect prompt pass ---
    def embed_hook(module, input, output):
        if not state["is_prompt_pass"]:
            state["generated_ids"].extend(input[0][0].tolist())

    # --- steer hooks: apply v * ALPHA to each target layer when flag is set ---
    last_steer_layer_idx = STEER_LAYERS[-1]

    def make_steer_hook(layer_idx):
        def hook(module, input, output):
            if state["is_prompt_pass"] or not state["should_steer"]:
                return output
            if state["steer_count"] > MAX_STEERS_PER_PROBLEM:
                return output
            hs = output[0]  # [1, 1, hidden_dim] during cached generation
            sv = steering_vec.to(hs.device).to(hs.dtype)
            new_hs = hs.clone()
            new_hs[0, -1, :] = hs[0, -1, :] + sv * ALPHA
            if layer_idx == last_steer_layer_idx:
                state["should_steer"] = False   # flag consumed after last steer layer
            return (new_hs,) + output[1:]
        return hook

    # --- probe hook: read from last layer, set steer flag at paragraph breaks ---
    def probe_hook(module, input, output):
        if state["is_prompt_pass"]:
            state["is_prompt_pass"] = False     # first firing = prompt pass done
            return output

        if state["steer_count"] >= MAX_STEERS_PER_PROBLEM:
            return output

        current_text = tokenizer.decode(state["generated_ids"], skip_special_tokens=False)
        new_suffix = current_text[len(state["prev_text"]):]
        if "\n\n" not in new_suffix:
            return output

        state["prev_text"] = state["prev_text"] + new_suffix.split("\n\n", 1)[0] + "\n\n"

        h_last = output[0][0, -1, :].float()
        conf   = probe_confidence(h_last)

        if conf < STEER_THRESHOLD:
            state["should_steer"] = True
            state["steer_count"]  += 1
            state["steer_log"].append({
                "token_pos":   len(state["generated_ids"]),
                "conf_before": round(conf, 4),
            })

        return output  # probe hook never modifies the hidden state

    # Register all hooks
    handles = []
    handles.append(model.model.embed_tokens.register_forward_hook(embed_hook))
    for li in STEER_LAYERS:
        handles.append(model.model.layers[li].register_forward_hook(make_steer_hook(li)))
    handles.append(model.model.layers[PROBE_LAYER].register_forward_hook(probe_hook))

    try:
        with torch.no_grad():
            gen = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        for h in handles:
            h.remove()

    generated_ids  = gen[0, prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False)
    n_tokens       = int(generated_ids.shape[0])

    return generated_text, n_tokens, state["steer_count"], state["steer_log"]


print(f"Steer layers: {STEER_LAYERS}  (probe reads from layer {28 + PROBE_LAYER if PROBE_LAYER < 0 else PROBE_LAYER})")

## Run steering on failed problems

In [ ]:
import json
from math_verify import parse, verify
from tqdm.auto import tqdm

results = []
n_correct_steered = 0
t_start = time.time()

for problem_idx in tqdm(failed_ids, desc="steering"):
    if problem_idx >= len(problems):
        print(f"  skip problem_idx={problem_idx}: out of dataset range")
        continue

    row    = problems.iloc[problem_idx]
    prompt = build_prompt(row["Question"])

    t0 = time.time()
    gen_text, n_tokens, n_steers, steer_log = generate_with_steering(prompt)
    wall = time.time() - t0

    gold = parse(f"${row['Answer']}$")
    pred = parse(gen_text)
    is_correct = bool(verify(gold, pred)) if pred else False
    if is_correct:
        n_correct_steered += 1

    record = {
        "problem_idx":   problem_idx,
        "correct_answer": str(row["Answer"]),
        "predicted":     str(pred[0]) if pred else "<none>",
        "label_steered": int(is_correct),
        "n_tokens":      n_tokens,
        "n_steers":      n_steers,
        "wall_s":        round(wall, 2),
        "steer_log":     steer_log,
    }
    results.append(record)

    print(
        f"  [{len(results)}/{len(failed_ids)}] idx={problem_idx:4d} "
        f"correct={is_correct} tokens={n_tokens} steers={n_steers} wall={wall:.1f}s"
    )

print(f"\nTotal wall time: {(time.time() - t_start) / 60:.1f} min")
print(f"Correct with steering: {n_correct_steered} / {len(results)} "
      f"({n_correct_steered / max(len(results), 1):.1%})")

## Aggregate results

In [ ]:
import numpy as np

n_total    = len(results)
n_correct  = sum(r["label_steered"] for r in results)
acc_steered = n_correct / max(n_total, 1)

token_counts = [r["n_tokens"] for r in results]
steer_counts = [r["n_steers"] for r in results]

summary = {
    "n_problems":          n_total,
    "n_correct_steered":   n_correct,
    "acc_steered":         round(acc_steered, 4),
    "acc_original":        0.0,                     # all these were originally incorrect
    "alpha":               ALPHA,
    "steer_threshold":     STEER_THRESHOLD,
    "tokens_mean":         round(float(np.mean(token_counts)), 1),
    "tokens_median":       round(float(np.median(token_counts)), 1),
    "tokens_max":          int(np.max(token_counts)),
    "steers_mean":         round(float(np.mean(steer_counts)), 2),
    "steers_total":        int(np.sum(steer_counts)),
    "per_problem":         results,
}

with open(STEERING_RESULTS_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Results -> {STEERING_RESULTS_PATH}")
print(f"\nSteered accuracy:  {n_correct}/{n_total}  ({acc_steered:.1%})")
print(f"Original accuracy: 0/{n_total}  (0.00%)  [all were failures]")
print(f"\nToken usage (problems that failed originally):")
print(f"  mean:   {summary['tokens_mean']:.0f}")
print(f"  median: {summary['tokens_median']:.0f}")
print(f"  max:    {summary['tokens_max']}")
print(f"\nSteering interventions:")
print(f"  total:  {summary['steers_total']}")
print(f"  mean/problem: {summary['steers_mean']:.2f}")

## Per-steer confidence breakdown

In [ ]:
all_steers = [s for r in results for s in r["steer_log"]]

if all_steers:
    confs_before = [s["conf_before"] for s in all_steers]
    print(f"Confidence at steer events: mean={np.mean(confs_before):.4f}  "
          f"min={np.min(confs_before):.4f}  max={np.max(confs_before):.4f}")

    # Problems where steering fired at least once
    steered_problems     = [r for r in results if r["n_steers"] > 0]
    n_steered            = len(steered_problems)
    n_steered_correct    = sum(r["label_steered"] for r in steered_problems)
    n_not_steered        = len(results) - n_steered
    n_not_steered_correct = sum(r["label_steered"] for r in results if r["n_steers"] == 0)

    print(f"\nProblems where steering fired (≥1 intervention):")
    print(f"  {n_steered} problems, {n_steered_correct} correct "
          f"({n_steered_correct / max(n_steered, 1):.1%})")
    print(f"Problems where steering never fired:")
    print(f"  {n_not_steered} problems, {n_not_steered_correct} correct "
          f"({n_not_steered_correct / max(n_not_steered, 1):.1%})")
else:
    print("No steering interventions occurred. Try lowering STEER_THRESHOLD or increasing ALPHA.")